In [16]:
!nvidia-smi

Sun Jun 21 19:27:46 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.106.00   Driver Version: 460.106.00   CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  A100-PCIE-40GB      Off  | 00000000:AF:00.0 Off |                    0 |
| N/A   63C    P0    54W / 250W |   3644MiB / 40536MiB |     37%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [1]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings

# Use float32 (Single Precision)
TORCH_DTYPE = torch.float32
NP_DTYPE = np.float32

# Torch defaults initialization 
torch.set_default_dtype(TORCH_DTYPE)
warnings.filterwarnings("ignore", category=FutureWarning)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

# ============================================================
# Reproducibility Setup
# ============================================================
base_seed = 2025
np.random.seed(base_seed)
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# Data Loading & Verification (Ensure correct names/precision)
# ============================================================
# Load clean reference for valuation
data_clean = np.load("lorenz63_noise_0.npy", allow_pickle=True).item()
l63 = data_clean["data"].T.astype(NP_DTYPE) 

# Load noisy reference for input
data_noisy = np.load("lorenz63_noise_10.npy", allow_pickle=True).item()
X_noisy_dataset = data_noisy["data"].T.astype(NP_DTYPE)

# Cast variables safely to tensors
X = torch.tensor(X_noisy_dataset, dtype=TORCH_DTYPE)
X_true = torch.tensor(l63, dtype=TORCH_DTYPE)

# Dataset splits
warmup_len, train_len, val_len, test_len = 100, 3900, 500, 500

X_warmup = X[:warmup_len].to(device)
X_train_clean = X[warmup_len:warmup_len + train_len]
X_val  = X[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test = X[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

X_val_true  = X_true[warmup_len + train_len : warmup_len + train_len + val_len].to(device)
X_test_true = X_true[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len].to(device)

train_noise_scale = 0.0  
X_train = X_train_clean.clone().to(device)


# ============================================================
# Adaptive NVAR Model Components (Double Precision Support)
# ============================================================
class FeatureMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, dtype=TORCH_DTYPE),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim, dtype=TORCH_DTYPE)
        )

    def forward(self, x):
        return self.net(x)


class AdaptiveNVARModel(nn.Module):
    def __init__(self, dk, m, d, hidden_dim):
        super().__init__()
        self.mlp = FeatureMLP(dk, hidden_dim, m)
        self.readout = nn.Linear(dk + m, d, bias=True, dtype=TORCH_DTYPE)

    def forward(self, H_lin):
        H_nn = self.mlp(H_lin)
        H_total = torch.cat([H_lin, H_nn], dim=1)
        return self.readout(H_total)


# ============================================================
# Helper Functions
# ============================================================
def construct_H_lin(X, k):
    """
    Build delay vectors consistent with NG-RC:

    [x(t), x(t-1), ..., x(t-k+1)]
    """

    T = X.shape[0]
    H = []

    for t in range(k - 1, T - 1):
        delays = [
            X[t - delay]
            for delay in range(k)
        ]

        H.append(torch.cat(delays, dim=0))

    return torch.stack(H)


def init_weights_stable(m):
    """Custom initialization optimized for Adaptive NVAR."""
    if isinstance(m, nn.Linear):
        if m.bias is None:
            nn.init.normal_(m.weight, mean=0.0, std=1e-4)
        else:
            nn.init.xavier_normal_(m.weight, gain=nn.init.calculate_gain('tanh'))
            nn.init.zeros_(m.bias)


# ============================================================
# Training Function (AdamW Optimization)
# ============================================================
def train_joint_model(
    X, k, m, hidden_dim=200,
    lr_adam=1e-4, max_epochs_adam=10000,
    adam_patience=500, tolerance=1e-5,
    device=None
):
    """Train Adaptive NVAR model using optimized AdamW with early stopping."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X = X.to(device=device, dtype=TORCH_DTYPE)

    H_lin = construct_H_lin(X, k)       # [n, dk]
    Y = X[k:] - X[k-1:-1]               # [n, d]
    dk = H_lin.shape[1]
    d = Y.shape[1]

    model = AdaptiveNVARModel(dk, m, d, hidden_dim).to(device)
    model.apply(init_weights_stable)

    # Use AdamW for direct parameter regularization
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_adam, weight_decay=1e-4)
    
    # Sharp, fast scheduler adjustments for trajectory resolution
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.2, patience=30
    )

    best_loss = float("inf")
    epochs_no_improve = 0
    in_memory_state = None
    
    for epoch in range(max_epochs_adam):
        model.train()
        Y_hat = model(H_lin)
        loss = F.mse_loss(Y_hat, Y)

        optimizer.zero_grad()
        loss.backward()
        
        # Clip gradient norms to stabilize chaotic loss trajectories
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step(loss.item())

        if best_loss - loss.item() > tolerance:
            best_loss = loss.item()
            epochs_no_improve = 0
            in_memory_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= adam_patience:
            break

    if in_memory_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in in_memory_state.items()})

    return model


# ============================================================
# Evaluation Function (Device & Precision Fixed)
# ============================================================
def evaluate_model(
    model,
    k,
    X_val_true,
    X,
    warmup_len,
    train_len,
    num_windows,
    horizons,
    device
):
    window_size = int(len(X_val_true) // num_windows)

    horizon_rmses = {h: [] for h in horizons}

    model.eval()

    for w in range(num_windows):

        start = w * window_size
        end   = (w + 1) * window_size

        X_val_true_window = X_val_true[start:end].to(
            device=device,
            dtype=TORCH_DTYPE
        )

        # --------------------------------------------------
        # Initial history
        # --------------------------------------------------
        init_start = warmup_len + train_len + start - k
        init_end   = warmup_len + train_len + start

        X_init = X[init_start:init_end].to(
            device=device,
            dtype=TORCH_DTYPE
        )

        x_t = [x.clone() for x in X_init.unbind(0)]

        predictions = []

        # --------------------------------------------------
        # Autoregressive rollout
        # --------------------------------------------------
        for _ in range(window_size):

            # NG-RC-consistent ordering:
            # [x(t), x(t-1), ..., x(t-k+1)]
            H_lin = torch.cat(
                [x_t[-1 - delay] for delay in range(k)],
                dim=0
            ).unsqueeze(0)

            with torch.no_grad():
                delta_x = model(H_lin)

            x_next = x_t[-1] + delta_x.squeeze(0)

            predictions.append(x_next)

            x_t = x_t[1:] + [x_next]

        predictions_np = (
            torch.stack(predictions)
            .cpu()
            .numpy()
            .astype(NP_DTYPE)
        )

        y_true_np = (
            X_val_true_window
            .cpu()
            .numpy()
            .astype(NP_DTYPE)
        )

        for h in horizons:
            if h <= window_size:

                rmse = np.sqrt(
                    np.mean(
                        (y_true_np[:h] - predictions_np[:h]) ** 2
                    )
                )

                horizon_rmses[h].append(rmse)

    return {
        h: np.mean(horizon_rmses[h])
        for h in horizons
    }

# ============================================================
# Optuna Objective
# ============================================================
def objective(trial):
    # Optimized search boundaries based on your low-noise tuning landmarks
    k = trial.suggest_categorical("k", [2, 5, 10, 20, 30, 40, 50])
    hidden_dim = trial.suggest_categorical("hidden_dim", [8, 32, 128, 256, 512])
    lr_adam = trial.suggest_categorical("lr_adam", [1e-4, 1e-3, 1e-2])
    
    dk = d * k
#    m = dk * (dk + 1) // 2
    m = 10 * dk
    all_run_rmses = {h: [] for h in horizons}

    for run in range(num_runs):
        seed = base_seed + run

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        model = train_joint_model(
            X_train, k, m, hidden_dim, lr_adam, device=device
        )

        rmse_by_horizon = evaluate_model(
            model, k, X_val_true, X, warmup_len, train_len, num_windows, horizons, device
        )

        for h in horizons:
            all_run_rmses[h].append(rmse_by_horizon[h])

    avg_rmse_horizons = {h: np.mean(all_run_rmses[h]) for h in horizons}
    trial.set_user_attr("rmse_horizons", avg_rmse_horizons)

    all_results.append({
        "k": k, "hidden_dim": hidden_dim, "lr_adam": lr_adam, "rmse": avg_rmse_horizons
    })

    print(
        f"[Trial {trial.number:03d}] k={k:2d} | hd={hidden_dim:3d} | adam={lr_adam:.2e} | "
        + " | ".join(f"h{h}={avg_rmse_horizons[h]:.6f}" for h in horizons)
    )

    return avg_rmse_horizons[100]


# ============================================================
# Configuration / Run Setup Block
# ============================================================
if __name__ == "__main__":
    num_runs = 1
    num_windows = 5
    d = 3
    horizons = [25, 50, 75, 100]
    all_results = []

    print("Starting Optuna...\n")

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=base_seed)
    )

    study.optimize(objective, n_trials=10, show_progress_bar=True)

    # ============================================================
    # Results Presentation
    # ============================================================
    best_trial = study.best_trial
    best_params = best_trial.params

    print("\n" + "=" * 60)
    print("BEST CONFIGURATION FOUND")
    print("=" * 60)
    print(f"k          : {best_params['k']}")
    print(f"hidden_dim : {best_params['hidden_dim']}")
    print(f"lr_adam    : {best_params['lr_adam']:.6e}")
    print(f"RMSE@100   : {best_trial.value:.6f}")

    print("\nRMSE by Horizon")
    print("-" * 60)
    for h, rmse in best_trial.user_attrs["rmse_horizons"].items():
        print(f"h={h:<3d} : {rmse:.6f}")

/home/eric/.conda/envs/nvar/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device in use: cuda


[I 2026-06-20 22:43:30,777] A new study created in memory with name: no-name-da639f7c-9c13-4e5d-ab0d-f1f22ea6d275


Starting Optuna...



Best trial: 0. Best value: 0.413246:  10%|████▋                                          | 1/10 [00:14<02:11, 14.60s/it]

...

[Trial 007] k= 2 | hd=128 | adam=1.00e-03 | h25=0.296469 | h50=0.497232 | h75=0.658755 | h100=0.694710
[I 2026-06-20 22:45:14,071] Trial 7 finished with value: 0.6947100162506104 and parameters: {'k': 2, 'hidden_dim': 128, 'lr_adam': 0.001}. Best is trial 4 with value: 0.1044921725988388.


Best trial: 4. Best value: 0.104492:  90%|██████████████████████████████████████████▎    | 9/10 [02:00<00:12, 12.86s/it]

[Trial 008] k=30 | hd=128 | adam=1.00e-03 | h25=0.069976 | h50=0.110555 | h75=0.152506 | h100=0.201340
[I 2026-06-20 22:45:31,180] Trial 8 finished with value: 0.20133952796459198 and parameters: {'k': 30, 'hidden_dim': 128, 'lr_adam': 0.001}. Best is trial 4 with value: 0.1044921725988388.


Best trial: 4. Best value: 0.104492: 100%|██████████████████████████████████████████████| 10/10 [02:05<00:00, 12.59s/it]

[Trial 009] k= 2 | hd=128 | adam=1.00e-03 | h25=0.296469 | h50=0.497232 | h75=0.658755 | h100=0.694710
[I 2026-06-20 22:45:36,673] Trial 9 finished with value: 0.6947100162506104 and parameters: {'k': 2, 'hidden_dim': 128, 'lr_adam': 0.001}. Best is trial 4 with value: 0.1044921725988388.

BEST CONFIGURATION FOUND
k          : 40
hidden_dim : 128
lr_adam    : 1.000000e-02
RMSE@100   : 0.104492

RMSE by Horizon
------------------------------------------------------------
h=25  : 0.088243
h=50  : 0.096966
h=75  : 0.099025
h=100 : 0.104492
